In [2]:
import torch 
import numpy as np
from chronos import ChronosPipeline
import yfinance as yf

In [1]:
TICKERS = {
    
    "Amazon": "AMZN",
    "Intel": "INTC",
    "AMD": "AMD",
    "General Electric": "GE",
}

In [9]:
def load_data(company_name, period="10y"):
    ticker = TICKERS[company_name]
    data = yf.download(ticker, period=period)
    data.reset_index(inplace=True)
    return  data
data = load_data("AMD")
price = data["Close"]

[*********************100%***********************]  1 of 1 completed


In [10]:
def chronos_predict(close_prices, months=6):
    pipeline = ChronosPipeline.from_pretrained(
        "amazon/chronos-t5-small",  # version légère
        device_map="cpu",
        torch_dtype=torch.float32,
    )

    context = torch.tensor(close_prices[-200:], dtype=torch.float32)
    n_days = months * 21

    forecast = pipeline.predict(
        context.unsqueeze(0),
        prediction_length=n_days,
        num_samples=20,
    )

    # Médiane des prédictions
    median_forecast = forecast[0].median(dim=0).values.numpy()
    return median_forecast

predicted_prices = chronos_predict(price.squeeze().to_numpy())
print(predicted_prices)

We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 


[256.47327 254.99937 256.47327 253.5253  254.99937 254.99937 253.5253
 252.0514  252.0514  249.10344 241.73343 235.83751 237.31158 240.25955
 240.25955 234.36362 231.41565 231.41565 225.51971 215.20174 215.20174
 219.62378 226.9936  228.46768 228.46768 231.41565 226.9936  221.09767
 226.9936  231.41565 226.9936  219.62378 221.09767 221.09767 219.62378
 213.72784 215.20174 209.3058  216.67581 210.77988 210.77988 218.1497
 219.62378 218.1497  219.62378 219.62378 215.20174 221.09767 225.51971
 222.57175 225.51971 224.04564 225.51971 224.04564 224.04564 222.57175
 224.04564 224.04564 235.83751 229.94157 234.36362 224.04564 226.9936
 225.51971 225.5309  224.00714 224.00714 224.00714 224.00714 224.00714
 222.48318 220.95943 220.95943 224.00714 224.00714 222.48318 224.00714
 224.00714 224.00714 222.48318 220.95943 219.43547 222.48318 222.48318
 224.00714 227.05486 228.57861 228.57861 225.5309  225.5309  225.5309
 224.00714 225.5309  225.5309  227.05486 231.62633 227.05486 227.05486
 230.10257

In [11]:
np.save("AMD_predicted_prices.npy", predicted_prices)